## Import Libraries

In [ ]:
# ── Notebook bootstrap: resolve the production module path ─────────────────
# This single cell replaces all duplicated production logic.
# Strategy: insert public/image-analysis-miu-batubara/ into sys.path so that
#   `import circle_detection` and `import block_detection` resolve to the
#   exact same source files served by the PyScript production environment.
# The path is computed from the repo root, found by walking up from this
#   notebook file.  VS Code exposes __vsc_ipynb_file__; for other kernels we
#   fall back to Path.cwd() (reliable when the server is started from the root).

import sys
from pathlib import Path

def _find_repo_root() -> Path:
    """Walk up from the notebook location until we find pytest.ini (repo marker)."""
    # VS Code Jupyter sets __vsc_ipynb_file__ to the notebook's absolute path.
    try:
        start = Path(__vsc_ipynb_file__).resolve().parent  # noqa: F821
    except NameError:
        start = Path.cwd()  # fallback: assumes kernel was started from repo root

    for candidate in [start, *start.parents]:
        if (candidate / "pytest.ini").exists():
            return candidate
    # Last resort: return start (notebook dir / CWD)
    return start

_REPO_ROOT = _find_repo_root()
_APP_DIR = str(_REPO_ROOT / "public" / "image-analysis-miu-batubara")

if _APP_DIR not in sys.path:
    sys.path.insert(0, _APP_DIR)

print(f"✅ Production module path: {_APP_DIR}")
print(f"   Resolved repo root   : {_REPO_ROOT}")


In [ ]:
# ── Production imports (Single Source of Truth) ────────────────────────────
# All detection logic lives exclusively in circle_detection.py.
# No code is duplicated here.
from circle_detection import (
    process_tiff_image,
    detect_grid_from_diagonal,
    analyze_grid_histograms,
    visualize_circle_invalid_roi,
    compare_diagonals,
    AIR_CV_THRESHOLD,
    AIR_DIAGONAL_VALIDATION_CODE,
    AIR_DIAGONAL_VALIDATION_ERROR,
)
print("✅ circle_detection imported from production source.")


## Define Image Processing Function

In [ ]:
# ── Notebook-only helpers (not in production) ──────────────────────────────
# These thin wrappers let notebooks pass a file *path* (str/Path) in addition
# to raw bytes, mirroring the convenience expected in an analysis workflow.

from pathlib import Path


def _to_file_bytes(file_or_bytes):
    if isinstance(file_or_bytes, (bytes, bytearray)):
        return bytes(file_or_bytes)
    if isinstance(file_or_bytes, Path):
        return file_or_bytes.read_bytes()
    if isinstance(file_or_bytes, str):
        return Path(file_or_bytes).read_bytes()
    raise TypeError(f"Unsupported input type: {type(file_or_bytes)!r}")


def process_image(image_path, **params):
    """Notebook convenience: accepts a file path as well as raw bytes."""
    return process_tiff_image(_to_file_bytes(image_path), params)


def run_full_miu_analysis(image_path, detect_params=None, compare_params=None):
    """End-to-end production-identical circle MIU pipeline for notebook debugging."""
    file_bytes = _to_file_bytes(image_path)
    detected  = process_tiff_image(file_bytes, detect_params or {})
    grid      = detect_grid_from_diagonal(file_bytes, detected)
    analysis  = compare_diagonals(file_bytes, grid, params=compare_params or {})
    return {
        "detected": detected,
        "grid":     grid,
        "analysis": analysis,
    }


print("✅ Notebook helpers defined (process_image, run_full_miu_analysis).")


## Process Image

Specify the image file path and run the processing function.

In [ ]:
image_file = r"sample-circle.tiff"
process_image(image_file)

## Run Full MIU Analysis (Production Logic)

Run the production-identical circle pipeline end-to-end: detection, grid extrapolation, and differential attenuation (MIU) analysis.

In [ ]:
# End-to-end run using the inlined production functions in Cell 4
image_file = r"sample-circle.tiff"

# Use integration-tested params for stable 4x4 anchors on sample-circle.tiff
detect_params = {
    "threshold_value": 24000,
    "min_diameter": 280,
    "max_diameter": 340,
    "min_circularity": 0.6,
    "min_solidity": 0.7,
    "expected_count": 16,
    "grid_cols": 4,
}

# Keep same tolerance used in integration tests
AIR_CV_THRESHOLD = 0.06

miu_result = run_full_miu_analysis(
    image_file,
    detect_params=detect_params,
    compare_params={"air_cv_threshold": AIR_CV_THRESHOLD},
)

print("=== MIU Summary ===")
summary = miu_result["analysis"]["summary"]
print(f"upper_mu_avg : {summary['upper_mu_avg']:.5f}")
print(f"lower_mu_avg : {summary['lower_mu_avg']:.5f}")
print(f"upper_mu_std : {summary['upper_mu_std']:.5f}")
print(f"lower_mu_std : {summary['lower_mu_std']:.5f}")
print(f"upper_mu_final: {summary['upper_mu_final']:.5f}")
print(f"lower_mu_final: {summary['lower_mu_final']:.5f}")

# Keep compatibility variable names for downstream cells
grid_results = miu_result["grid"]
comparison_result = miu_result["analysis"]

## Histogram Analysis (Production Function)

Use the production histogram analyzer over the extrapolated 4x4 grid.

In [ ]:
# Run production histogram analyzer (no function override)
if "grid_results" in locals():
    hist_result = analyze_grid_histograms(image_file, grid_results)
    if hist_result is None:
        print("Histogram analysis returned no result")
    else:
        print("Histogram stats count:", len(hist_result["histogram_stats"]))
        print("ROI area mean:", f"{hist_result['roi_area_mean']:.3f}")
        print("ROI area std :", f"{hist_result['roi_area_std']:.3f}")
else:
    print("Run the MIU analysis cell first.")

## Differential Attenuation (MIU) Output

Display key attenuation metrics from the production compare function output.

In [ ]:
# Show detailed production MIU outputs (already computed in miu_result)
if "comparison_result" in locals():
    s = comparison_result["summary"]
    print("=== Differential Attenuation Summary ===")
    print(f"p_air          : {s['p_air']:.5f}")
    print(f"x_coal_mm      : {s['x_coal_mm']:.5f}")
    print(f"anti_air_cv    : {s['anti_air_cv']:.5f}")
    print(f"upper_mu_avg   : {s['upper_mu_avg']:.5f}")
    print(f"lower_mu_avg   : {s['lower_mu_avg']:.5f}")
    print(f"upper_mu_std   : {s['upper_mu_std']:.5f}")
    print(f"lower_mu_std   : {s['lower_mu_std']:.5f}")
    print(f"upper_mu_final : {s['upper_mu_final']:.5f}")
    print(f"lower_mu_final : {s['lower_mu_final']:.5f}")

    print("\nUpper stats count:", len(comparison_result["upper_stats"]))
    print("Lower stats count:", len(comparison_result["lower_stats"]))
else:
    print("Run the MIU analysis cell first.")